# 03c — SHAP Local: giải thích từng khách hàng

**Mục đích:** Tài liệu này trình bày giải thích cục bộ (local explanation) bằng SHAP cho từng khách hàng, gồm Waterfall Plot theo biến gốc, Waterfall Plot theo nhóm đặc trưng (cơ sở của reason codes trên demo) và Force Plot, cho ba trường hợp đại diện: **Bình thường**, **Cảnh báo cao** và **Ca biên** (Tuần 2 – T5).

**Phạm vi:**

- **Mô hình:** LightGBM + monotonic constraints với bộ tham số tốt nhất của Optuna (`configs/tuned_params.yaml`), ứng viên chính cho mô hình cuối (`reports/experiments_optuna_tuning.md` mục 7). Việc chọn mô hình cuối thuộc buổi họp review Tuần 2 (T6); notebook chạy lại được cho mô hình khác bằng cách đổi `STUDY`.
- **Dữ liệu:** toàn bộ tập Valid (6.000 khách hàng). Tập Test không được sử dụng.
- **Thang giải thích:** SHAP trên thang margin (log-odds) của mô hình chưa hiệu chuẩn. Sau khi Platt scaling được fit ở Tuần 3 – T2 (PD = sigmoid(a·margin + b)), giá trị SHAP được nhân với hệ số a để chuyển sang thang logit của PD đã hiệu chuẩn (project_plan mục 3.5e); thứ tự và chiều của các đóng góp không đổi.
- **Thuật ngữ:** plan Tuần 2 – T5 gọi ba trường hợp là Duyệt, Từ chối và Ca biên. Do mô hình được dùng làm công cụ cảnh báo sớm trên danh mục hiện hữu, không phải công cụ xét duyệt (Model Card), notebook dùng thuật ngữ của demo: Bình thường (tương ứng Duyệt), Cảnh báo cao (tương ứng Từ chối) và Ca biên.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_split import load_split_data
from src.explain import aggregate_by_group, aggregate_by_variable, load_feature_groups, reason_codes
from src.features import build_features
from src.scoring import pd_to_score
from src.tune import fit_tuned_pipeline

STUDY = "lightgbm_monotone"
THRESHOLD = 0.30          # ngưỡng tạm thời của demo (app/streamlit_app.py); ngưỡng chính thức chốt ở Tuần 3 – T2
MAX_DISPLAY = 10
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 20)
# shap 0.46 cảnh báo thay đổi định dạng đầu ra với LightGBM nhị phân; notebook đọc đúng dạng ndarray (n, số cột).
warnings.filterwarnings("ignore", message="LightGBM binary classifier with TreeExplainer")
print("shap", shap.__version__)

## 1. Mô hình và giá trị SHAP

Mô hình được fit trên toàn bộ tập Train bằng `fit_tuned_pipeline`, cùng tham số và `random_state` với mô hình đã đăng ký `lightgbm_monotone_tuned` trên MLflow. `TreeExplainer` tính SHAP chính xác cho mô hình cây; tính cộng tính được kiểm tra trên thang margin: $\phi_0 + \sum_i \phi_i = m(x)$, với $m(x)$ là đầu ra `raw_score=True` của LightGBM.

Giá trị SHAP của các cột one-hot được cộng về biến gốc (`aggregate_by_variable`), sau đó cộng theo năm nhóm đặc trưng khai báo trong `configs/config.yaml` (`aggregate_by_group`). Do SHAP có tính cộng tính, tổng đóng góp ở cả ba cấp (cột, biến, nhóm) đều bằng nhau.

In [ ]:
train_df, valid_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_valid, y_valid = build_features(valid_df)

t0 = time.perf_counter()
pipe = fit_tuned_pipeline(STUDY, X_train, y_train)
print(f"Fit trên Train: {time.perf_counter() - t0:.1f} giây")

prep, model = pipe.named_steps["preprocess"], pipe.named_steps["clf"]
col_tf = prep.named_steps["preprocessor"]
columns = [name.split("__", 1)[1] for name in col_tf.get_feature_names_out()]
cat_vars = list(col_tf.transformers_[0][2])

Z_valid = prep.transform(X_valid)
explainer = shap.TreeExplainer(model)
t0 = time.perf_counter()
phi = explainer.shap_values(Z_valid)
elapsed = time.perf_counter() - t0
base = float(np.ravel(explainer.expected_value)[-1])

margin = model.predict(Z_valid, raw_score=True)
pd_valid = 1 / (1 + np.exp(-margin))
err_margin = np.abs(base + phi.sum(axis=1) - margin).max()
err_proba = np.abs(pd_valid - pipe.predict_proba(X_valid)[:, 1]).max()

groups = load_feature_groups()
shap_by_var = aggregate_by_variable(phi, columns, cat_vars)
shap_by_group = aggregate_by_group(shap_by_var, groups)
err_group = np.abs(base + shap_by_group.sum(axis=1).to_numpy() - margin).max()

print(f"SHAP cho {len(X_valid):,} khách hàng × {len(columns)} cột: {elapsed:.1f} giây")
print(f"Base value φ0 = {base:.3f} (margin), tương ứng PD = {1 / (1 + np.exp(-base)):.3f}")
print(f"Sai số cộng tính lớn nhất — cột: {err_margin:.1e}, nhóm: {err_group:.1e}; PD so với predict_proba: {err_proba:.1e}")
assert err_margin < 1e-6 and err_group < 1e-6

## 2. Chọn trường hợp minh họa

Ba trường hợp được chọn từ tập Valid theo quy tắc cố định, nên kết quả tái lập được khi chạy lại:

| Trường hợp | Quy tắc chọn |
|---|---|
| Bình thường | Khách hàng không vỡ nợ có PD gần nhất với phân vị 10% của PD trên Valid |
| Cảnh báo cao | Khách hàng vỡ nợ có PD gần nhất với phân vị 95% của PD trên Valid |
| Ca biên | Khách hàng có PD gần nhất với ngưỡng cảnh báo tạm thời 0,30 (không phụ thuộc nhãn) |

Mức cảnh báo được gán theo quy tắc của demo: PD ≥ 0,30 là Cảnh báo, 0,15 ≤ PD < 0,30 là Theo dõi, còn lại là Bình thường. Điểm tín dụng quy đổi theo PDO (600 điểm tại odds 50:1, PDO = 20) từ PD chưa hiệu chuẩn, chỉ mang tính tham khảo.

In [ ]:
def risk_band(p, threshold=THRESHOLD):
    return "Cảnh báo" if p >= threshold else ("Theo dõi" if p >= threshold / 2 else "Bình thường")


res = pd.DataFrame({"pd": pd_valid, "y": y_valid.to_numpy()}, index=X_valid.index)
cases = {
    "Bình thường": (res.loc[res.y == 0, "pd"] - res.pd.quantile(0.10)).abs().idxmin(),
    "Cảnh báo cao": (res.loc[res.y == 1, "pd"] - res.pd.quantile(0.95)).abs().idxmin(),
    "Ca biên": (res.pd - THRESHOLD).abs().idxmin(),
}
pos = {name: X_valid.index.get_loc(idx) for name, idx in cases.items()}

frame = prep[:-1].transform(X_valid)  # đặc trưng trước encode/scale (giá trị gốc, có biến dẫn xuất và AGE_BIN)
show_cols = ["LIMIT_BAL", "PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6", "PAY_MAX", "UTIL_MEAN",
             "BILL_AMT1", "PAY_AMT1", "AGE_BIN"]
case_table = pd.DataFrame({
    name: {
        "dòng (index Valid)": idx,
        "PD": round(float(pd_valid[pos[name]]), 3),
        "điểm (PDO)": int(round(float(pd_to_score(pd_valid[pos[name]])))),
        "mức cảnh báo": risk_band(pd_valid[pos[name]]),
        "vỡ nợ thực tế": int(y_valid.iloc[pos[name]]),
        **{c: frame.iloc[pos[name]][c] for c in show_cols},
    }
    for name, idx in cases.items()
})
case_table

## 3. Waterfall Plot theo biến gốc

Waterfall Plot bắt đầu từ base value $E[f(X)]$ (margin trung bình của mô hình) ở cuối biểu đồ và cộng lần lượt đóng góp của từng biến để đến đầu ra $f(x)$ của khách hàng ở đỉnh biểu đồ. Thanh đỏ làm tăng margin (tăng PD), thanh xanh làm giảm. Biến phân loại được gộp từ các cột one-hot; nhãn trục tung là giá trị gốc của biến. Biểu đồ hiển thị 9 biến có |đóng góp| lớn nhất, phần còn lại được gộp vào một dòng.

In [ ]:
variables = list(shap_by_var.columns)
SLUG = {"Bình thường": "binh_thuong", "Cảnh báo cao": "canh_bao_cao", "Ca biên": "ca_bien"}


def display_value(value):
    if isinstance(value, (float, np.floating)):
        return round(float(value), 2)
    return value if isinstance(value, (int, np.integer)) else str(value)


def explanation_by_variable(i):
    return shap.Explanation(
        values=shap_by_var.iloc[i].to_numpy(),
        base_values=base,
        data=np.array([display_value(frame.iloc[i][v]) for v in variables], dtype=object),
        feature_names=variables,
    )


for name, i in pos.items():
    shap.plots.waterfall(explanation_by_variable(i), max_display=MAX_DISPLAY, show=False)
    plt.title(f"{name} — PD {pd_valid[i]:.3f}; SHAP trên thang margin")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"shap_local_waterfall_{SLUG[name]}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Reason codes theo nhóm đặc trưng

Do các biến trong cùng nhóm tương quan cao (vd `PAY_1`–`PAY_6`, `PAY_MAX`, `PAY_LATE_*`), SHAP phân bổ đóng góp giữa chúng khá tùy ý; reason codes vì vậy được tính trên nhóm đặc trưng (project_plan mục 3.5e). Mỗi trường hợp có một Waterfall Plot theo nhóm và bảng reason codes (`src.explain.reason_codes`): tối đa hai nhóm làm tăng rủi ro và hai nhóm làm giảm rủi ro nhiều nhất, bỏ qua nhóm có |đóng góp| ≤ 0,05 trên thang margin.

In [ ]:
group_names = list(shap_by_group.columns)
fig_rows = []
for name, i in pos.items():
    exp_group = shap.Explanation(values=shap_by_group.iloc[i].to_numpy(), base_values=base, feature_names=group_names)
    shap.plots.waterfall(exp_group, max_display=len(group_names), show=False)
    plt.title(f"{name} — đóng góp theo nhóm đặc trưng")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"shap_local_groups_{SLUG[name]}.png", dpi=150, bbox_inches="tight")
    plt.show()

    codes = reason_codes(shap_by_group.iloc[i], k=2, min_abs=0.05)
    codes.insert(0, "trường hợp", name)
    fig_rows.append(codes)

reason_table = pd.concat(fig_rows, ignore_index=True).round({"đóng góp": 3})
reason_table

## 5. Force Plot cho Ca biên

Force Plot thể hiện cùng thông tin với Waterfall Plot dưới dạng một trục ngang: các lực đỏ đẩy margin lên, các lực xanh kéo xuống, điểm cân bằng là đầu ra $f(x)$. Với Ca biên, hai nhóm lực có độ lớn gần nhau, nên Force Plot cho thấy trực quan vì sao khách hàng nằm sát ngưỡng.

In [ ]:
i = pos["Ca biên"]
shap.plots.force(
    base,
    shap_by_var.iloc[i].to_numpy(),
    features=np.array([display_value(frame.iloc[i][v]) for v in variables], dtype=object),
    feature_names=variables,
    matplotlib=True,
    show=False,
    contribution_threshold=0.08,
    figsize=(24, 3),
)
plt.savefig(FIG_DIR / "shap_local_force_ca_bien.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Tương tác `PAY_1` – `PAY_MAX` trên các khách hàng cùng mức trễ tối đa

Phân tích dependence (`03_shap_dependence.ipynb` mục 5) ghi nhận `PAY_1` – `PAY_MAX` là cặp tương tác mạnh nhất trên XGBoost và đề xuất lấy các hồ sơ có `PAY_MAX` = 2 nhưng `PAY_1` khác nhau làm minh họa cục bộ. Bảng dưới đây so sánh các khách hàng Valid có `PAY_MAX` = 2 theo trạng thái tháng gần nhất `PAY_1`, trên mô hình LightGBM + monotonic.

In [ ]:
mask = frame["PAY_MAX"] == 2
pay1_label = {-2: "-2 (không phát sinh dư nợ)", -1: "-1 (trả đủ)", 0: "0 (trả tối thiểu)", 1: "1 (trễ 1 tháng)",
              2: "2 (trễ 2 tháng)"}
interaction = (
    pd.DataFrame({
        "PAY_1": frame.loc[mask, "PAY_1"].to_numpy(),
        "PD": pd_valid[mask.to_numpy()],
        "vỡ nợ thực tế": y_valid[mask].to_numpy(),
        "SHAP PAY_1": shap_by_var.loc[mask.to_numpy(), "PAY_1"].to_numpy(),
        "SHAP PAY_MAX": shap_by_var.loc[mask.to_numpy(), "PAY_MAX"].to_numpy(),
        "SHAP nhóm trễ hạn": shap_by_group.loc[mask.to_numpy(), groups["lich_su_tre_han"]["label"]].to_numpy(),
    })
    .groupby("PAY_1")
    .agg(**{
        "số khách hàng": ("PD", "size"),
        "PD trung bình": ("PD", "mean"),
        "tỷ lệ vỡ nợ thực tế": ("vỡ nợ thực tế", "mean"),
        "SHAP PAY_1": ("SHAP PAY_1", "mean"),
        "SHAP PAY_MAX": ("SHAP PAY_MAX", "mean"),
        "SHAP nhóm trễ hạn": ("SHAP nhóm trễ hạn", "mean"),
    })
    .rename(index=pay1_label)
    .round(3)
)
interaction

## 7. Nhận xét

### 7.1. Tính đúng đắn và chi phí tính toán

- Sai số cộng tính lớn nhất ở mức $10^{-14}$ ở cả cấp cột và cấp nhóm; PD suy ra từ margin trùng với `predict_proba` của pipeline. Reason codes theo nhóm vì vậy cộng đúng về đầu ra của mô hình.
- Base value $\phi_0 = -1{,}562$ (margin), tương ứng PD ≈ 0,173, thấp hơn tỷ lệ vỡ nợ của Train (≈ 0,221). Base value là trung bình margin trên dữ liệu huấn luyện; do hàm sigmoid lồi ở vùng PD thấp, sigmoid của margin trung bình nhỏ hơn PD trung bình. Chênh lệch này không phải lỗi của mô hình.
- `TreeExplainer` tính SHAP cho 6.000 khách hàng trong khoảng 2 giây, tức dưới 1 ms cho một khách hàng, phù hợp cho giải thích trực tiếp trên demo (project_plan mục 7, rủi ro SHAP chậm trong demo).

### 7.2. Bình thường (PD 0,051; 571 điểm; không vỡ nợ)

- Cả năm nhóm đều làm giảm rủi ro. Nhóm mạnh nhất là **Lịch sử trễ hạn (−0,68)**: khách hàng không trễ hạn tháng nào trong 6 tháng (`PAY_1`–`PAY_6` = 0, trả mức tối thiểu đúng hạn), thể hiện qua `PAY_MAX` (−0,21), `PAY_1` (−0,20) và các biến `PAY_LATE_*`.
- Nhóm thứ hai là **Dư nợ và biến động dư nợ (−0,28)**. Hạn mức cao (280.000) cũng giúp giảm rủi ro qua nhóm Mức sử dụng hạn mức (−0,20).
- Không có nhóm nào làm tăng rủi ro vượt 0,05, nên reason codes của trường hợp này chỉ gồm các yếu tố giảm rủi ro.

### 7.3. Cảnh báo cao (PD 0,702; 462 điểm; vỡ nợ)

- **Lịch sử trễ hạn (+2,50) chi phối hoàn toàn**: khách hàng trễ 2 tháng liên tục từ `PAY_5` đến `PAY_1`. Đóng góp được chia cho `PAY_1` (+0,88), `PAY_MAX` (+0,58), `PAY_LATE_2PLUS_COUNT` (+0,38), `PAY_LATE_CONSECUTIVE` (+0,38) và `PAY_MEAN` (+0,21). Cách chia giữa các biến này phụ thuộc vào cấu trúc cây và không nên diễn giải riêng lẻ; tổng của nhóm là đại lượng ổn định.
- Bốn nhóm còn lại cộng lại chỉ khoảng −0,08. Với các hồ sơ rủi ro cao, reason codes thường chỉ có một nhóm chính.

### 7.4. Ca biên (PD 0,300; 512 điểm; không vỡ nợ)

- Khách hàng **từng trễ 2 tháng ở tháng 6 và tháng 7** (`PAY_4`, `PAY_3` = 2) nhưng **đã trả đủ hoặc không phát sinh dư nợ ở các tháng gần đây** (`PAY_2` = −1, `PAY_1` = −2). Hai xu hướng này cho ra hai nhóm lực gần cân bằng, thể hiện rõ trên Force Plot (mục 5):
  - **Tăng rủi ro:** Lịch sử trễ hạn (+0,52), qua `PAY_MAX` (+0,35), `PAY_LATE_CONSECUTIVE` (+0,23), `PAY_LATE_2PLUS_COUNT` (+0,22); Hành vi trả nợ (+0,20), do các tháng không thanh toán (`PAY_AMT3` = `PAY_AMT6` = 0, `PAY_RATIO_3` = 0); Dư nợ và biến động dư nợ (+0,18), chủ yếu do dư nợ tháng gần nhất `BILL_AMT1` = 0.
  - **Giảm rủi ro:** xu hướng trễ hạn đang cải thiện (`PAY_SLOPE` −0,15) và mức sử dụng hạn mức rất thấp (Mức sử dụng hạn mức −0,22).
- **`BILL_AMT1` = 0 làm tăng rủi ro** là hành vi khớp với dữ liệu, không phải lỗi mô hình: trên Train, khách hàng có dư nợ sao kê gần nhất bằng 0 có tỷ lệ vỡ nợ 27,8% (1.207 khách hàng), cao hơn 21,8% của nhóm có dư nợ dương. Tuy nhiên, đây là yếu tố dễ gây hiểu nhầm khi hiển thị ("dư nợ bằng 0 làm tăng rủi ro"), nên cần diễn đạt cẩn thận trên demo, vd "tài khoản không phát sinh giao dịch gần đây".
- Khách hàng thực tế không vỡ nợ. PD 0,300 nằm đúng ngưỡng cảnh báo tạm thời; mức cảnh báo cuối cùng của các hồ sơ như vậy phụ thuộc vào ngưỡng chốt ở Tuần 3 – T2.

### 7.5. Tương tác `PAY_1` – `PAY_MAX`

- Trong 1.432 khách hàng Valid có cùng `PAY_MAX` = 2, PD trung bình thay đổi từ 0,25 đến 0,67 theo trạng thái tháng gần nhất. Đóng góp của `PAY_MAX` ổn định (+0,35 đến +0,53), trong khi đóng góp của `PAY_1` thay đổi từ −0,34 (trả tối thiểu) đến +0,81 (vẫn trễ 2 tháng). Mô hình phân biệt khách hàng đã trả nợ trở lại với khách hàng vẫn đang trễ, xác nhận tương tác quan sát trên XGBoost (`03_shap_dependence.ipynb`) cũng có trên LightGBM + monotonic.
- PD trung bình bám sát tỷ lệ vỡ nợ thực tế của từng nhóm (vd 0,674 so với 0,683 khi `PAY_1` = 2; 0,410 so với 0,417 khi `PAY_1` = 1). Nhóm `PAY_1` = −2 chỉ có 13 khách hàng nên không đủ để kết luận.
- Trong nhóm này, khách hàng trả đủ ở tháng gần nhất (`PAY_1` = −1) có PD trung bình (0,302) và tỷ lệ vỡ nợ thực tế (0,315) cao hơn khách hàng trả tối thiểu (`PAY_1` = 0: 0,251 và 0,240). Thứ tự này khớp với quan hệ không đơn điệu ở vùng −2 … 0 đã ghi nhận tại `reports/experiments_optuna_tuning.md` mục 8.2, và là lý do các mã −2/−1/0 không được ép đơn điệu.

### 7.6. Hệ quả cho reason codes và demo

- Reason codes theo nhóm cho kết quả nhất quán với trực giác nghiệp vụ ở cả ba trường hợp. Với ngưỡng |đóng góp| > 0,05, trường hợp Bình thường chỉ có yếu tố giảm rủi ro, trường hợp Cảnh báo cao có một yếu tố tăng rủi ro chi phối, và Ca biên có cả hai chiều.
- Adapter hiện tại của demo (`app/baseline_model.py`) chỉ hỗ trợ Logistic Regression. Khi tích hợp mô hình cuối dạng cây ở Tuần 3 – T3, adapter cần dùng `TreeExplainer` như notebook này, kèm hệ số `a` của Platt scaling để chuyển SHAP sang thang logit của PD đã hiệu chuẩn.

### 7.7. Hạn chế

- SHAP được tính trên mô hình chưa hiệu chuẩn; PD, điểm và mức cảnh báo ở đây chỉ mang tính minh họa cho đến khi calibration và ngưỡng được chốt ở Tuần 3 – T2.
- `TreeExplainer` dùng phương pháp `tree_path_dependent`. Với các biến tương quan cao trong cùng nhóm, phần đóng góp chia cho từng biến phụ thuộc cấu trúc cây; giải thích cho người dùng chỉ nên hiển thị ở cấp nhóm.
- Ba trường hợp được chọn để minh họa theo quy tắc ở mục 2, không đại diện cho toàn bộ phân phối khách hàng.

## Hình xuất ra

Lưu tại `reports/figures/` (không commit, sinh lại bằng cách chạy notebook):

- `shap_local_waterfall_binh_thuong.png`, `shap_local_waterfall_canh_bao_cao.png`, `shap_local_waterfall_ca_bien.png` — Waterfall Plot theo biến gốc.
- `shap_local_groups_binh_thuong.png`, `shap_local_groups_canh_bao_cao.png`, `shap_local_groups_ca_bien.png` — Waterfall Plot theo nhóm đặc trưng.
- `shap_local_force_ca_bien.png` — Force Plot cho Ca biên.